In [15]:
!pip install onnx onnxruntime onnxruntime-tools

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time
import psutil
import os

# =========================
# ✅ MODEL DEFINITION
# =========================
class CustomCNN2(nn.Module):
    def __init__(self, num_classes=4):
        super(CustomCNN2, self).__init__()

        self.conv1 = nn.Conv2d(3, 256, 3)
        self.conv2 = nn.Conv2d(256, 128, 3)
        self.conv3 = nn.Conv2d(128, 64, 3)
        self.conv4 = nn.Conv2d(64, 32, 3)
        self.conv5 = nn.Conv2d(32, 8, 3)

        self.pool = nn.MaxPool2d(2, 2)

        # ✅ CORRECT (your working version)
        self.fc1 = nn.Linear(8, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = self.pool(F.relu(self.conv4(x)))
        x = self.pool(F.relu(self.conv5(x)))

        # x = x.view(x.size(0), -1)  # → 8
        x = torch.flatten(x, 1)
        print(x.shape)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x

In [17]:
!pip install onnxscript

^C


In [18]:


model = CustomCNN2()
model.load_state_dict(torch.load("Custommodel2_fp32.pth"))
model.eval()

dummy_input = torch.randn(1, 3, 110, 100)

torch.onnx.export(
    model,
    dummy_input,
    "model_fp32.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=11,
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}}
)

print("ONNX model saved!")

/tmp/ipykernel_5137/1319449062.py:7: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0419 15:33:51.023000 5137 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0419 15:33:52.051000 5137 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, al

[torch.onnx] Obtain model graph for `CustomCNN2([...]` with `torch.export.export(..., strict=False)`...
torch.Size([s77, 8])
[torch.onnx] Obtain model graph for `CustomCNN2([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 1 of general pattern rewrite rules.
ONNX model saved!


In [19]:
!pip install onnxsim

In [20]:
from onnxsim import simplify
import onnx

model = onnx.load("model_fp32.onnx")
model_simp, check = simplify(model)

onnx.save(model_simp, "model_fp32_simplified.onnx")

In [21]:
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    model_input="model_fp32_simplified.onnx",
    model_output="model_int8.onnx",
    weight_type=QuantType.QInt8
)

print("✅ INT8 ONNX model saved")

✅ INT8 ONNX model saved


LOADING TEST DATA

In [22]:
pip install gdown

In [23]:
# Download the zip file from Google Drive
import gdown
import os

# link: https://drive.google.com/file/d/1xNDwcGyhqY_7tdVc58YSJCS4DptDoEUL/view?usp=sharing
file_id = '1xNDwcGyhqY_7tdVc58YSJCS4DptDoEUL'
output_filename = 'archive.zip'

gdown.download(id=file_id, output=output_filename, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1xNDwcGyhqY_7tdVc58YSJCS4DptDoEUL
From (redirected): https://drive.google.com/uc?id=1xNDwcGyhqY_7tdVc58YSJCS4DptDoEUL&confirm=t&uuid=fcb42d6e-7c37-40a9-81dc-2981b36bbb0c
To: /content/archive.zip
100%|██████████| 1.27G/1.27G [00:17<00:00, 73.2MB/s]


'archive.zip'

In [24]:
# Extract the contents of the zip file
import zipfile

if os.path.exists(output_filename):
    with zipfile.ZipFile(output_filename, 'r') as zip_ref:
        zip_ref.extractall('.') # Extract to the current directory
    print(f"Successfully extracted '{output_filename}'")
    # Optionally, remove the zip file after extraction
    # os.remove(output_filename)
else:
    print(f"Error: The file '{output_filename}' was not found.")

KeyboardInterrupt: 

In [ ]:
# List the contents of the current directory to see extracted files
print("Contents of the current directory after extraction:")
print(os.listdir('.'))

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

transform=transforms.Compose([
    transforms.Resize((110, 100)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

full_dataset = datasets.ImageFolder(root='train', transform=transform)

In [ ]:
test_dataset = datasets.ImageFolder(root='test', transform=transform)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

In [ ]:
import onnxruntime
import numpy as np
from tqdm import tqdm

# Create an ONNX Runtime session
# Since it's an INT8 model, we can try to use CPUExecutionProvider for simplicity
sess_options = onnxruntime.SessionOptions()
# Optional: Set graph optimization level to enable more optimizations
sess_options.graph_optimization_level = onnxruntime.GraphOptimizationLevel.ORT_ENABLE_EXTENDED

sess = onnxruntime.InferenceSession("model_int8.onnx", sess_options=sess_options, providers=['CPUExecutionProvider'])

# Get input and output names
input_name = sess.get_inputs()[0].name
output_name = sess.get_outputs()[0].name

correct = 0
total = 0

print("Starting evaluation of the INT8 ONNX model...")

# Iterate over the test dataset
for inputs, labels in tqdm(test_loader, desc="Evaluating model"):
    # ONNX Runtime expects numpy arrays, so convert PyTorch tensors
    inputs_np = inputs.numpy()

    # Run inference
    outputs = sess.run([output_name], {input_name: inputs_np})

    # The output is a list, get the first (and usually only) element
    predictions = outputs[0]

    # Get predicted class (index of the max log-probability)
    predicted_labels = np.argmax(predictions, axis=1)

    total += labels.size(0)
    correct += (predicted_labels == labels.numpy()).sum().item()

accuracy = 100 * correct / total
print(f"\nAccuracy of the INT8 ONNX model on the test dataset: {accuracy:.2f}%")